In [0]:
# creating a dataframe
customers_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/ecomm/landing/operational_data1/customers_autoloader/_schema")
        .load("/Volumes/ecomm/landing/operational_data1/customers_autoloader/")
)

Here we are not goign to define the schema. Autoloader infers the schema

In [0]:
# we can use infer schema for it to infer schema
customers_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/ecomm/landing/operational_data1/customers_autoloader/_schema")
        .option("cloudFiles.inferColumnTypes", "True")
        .load("/Volumes/ecomm/landing/operational_data1/customers_autoloader/")
)

In [0]:
# we can define explicit schmea using schema hints
customers_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/ecomm/landing/operational_data1/customers_autoloader/_schema")
        .option("cloudFiles.inferColumnTypes", "True")
        .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
        # .option("pathGlobFilter", "customers_2024_*.json") -- to filter out the data file that we need
        .load("/Volumes/ecomm/landing/operational_data1/customers_autoloader/")
)

In [0]:
# basic transformations and stored into another dataframe
from pyspark.sql.functions import col
customer_transform_df = customers_df.withColumn("filepath", col("_metadata"))


In [0]:
# write stream and load into delta table

streaming_query = (customer_transform_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/ecomm/landing/operational_data1/customers_autoloader/_checkpoint_stream")
                        .toTable("ecomm.bronze.customers_autoloader")
                        )

# streaming_query.stop() to stop the streaming process.                         

In [0]:
%sql
select * from ecomm.bronze.customers_autoloader

created_timestamp,customer_id,customer_name,date_of_birth,email,member_since,telephone,_rescued_data,filepath
2024-10-17T16:12:27Z,9179,Richard Cox,1996-10-25,devon84@mail.com,2024-09-26,+1 6680703335,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-01T00:50:29Z,4858,Carla Morton,2004-06-21,joseph88@mail.com,2024-09-15,+1 8616454195,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-23T22:03:08Z,7207,Billy Scott,1997-03-17,christopher30@mail.com,2024-09-23,+1 5544387564,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-12T06:02:27Z,8539,Lori Mason,2002-11-01,stephanie7@mail.com,2024-09-12,+1 0498301620,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-24T13:03:13Z,9706,Jennifer Haas,2001-04-03,benjamin55@mail.com,2024-10-05,+1 4725460000,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-08T22:49:25Z,9263,Joseph Keller,2003-02-11,null,2024-10-04,+1 3817867756,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-06T19:55:52Z,5028,Jessica Harris,2004-04-19,null,2024-09-10,+1 8604009935,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-18T23:24:52Z,9018,William Carter,2003-09-05,james70@gmail.com,2024-10-08,+1 1448753611,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-21T13:20:26Z,8580,Shannon Austin,2002-03-22,john30@gmail.com,2024-10-07,+1 4594705629,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"
2024-10-02T14:53:40Z,3409,Andrew Phillips,2003-04-17,peter73@yahoo.com,2024-09-30,+1 4079273853,null,"List(/Volumes/ecomm/landing/operational_data1/customers_autoloader/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T21:13:34Z)"


each time When we upload a new file, there will be a spike in the graph

In [0]:
# we can define explicit schmea using schema hints
customers_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/ecomm/landing/operational_data1/customers_autoloader/_schema")
        .option("cloudFiles.inferColumnTypes", "True")
        .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")

        #.option("cloudFiles.schemaEvolutionMode", "rescue") -- handles unexpected columns. 
        # streaming runs, but the unexpected columns go to _rescued_data

        #.option("cloudFiles.schemaEvolutionMode", "addNewColumns")

        #.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")

        # .option("cloudFiles.schemaEvolutionMode", "none") -- to ignore or drop new columns
        .load("/Volumes/ecomm/landing/operational_data1/customers_autoloader/")
)

In [0]:
streaming_query.stop()